In [1]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
    
from utils.analyse_results import GraphsTraining, Results

train_info = GraphsTraining()

### Load Pyomo results in

In [2]:
fn = 'results/2024-08-21_09-45-20_training_convergence.pkl' # 16 neurons
fn = 'results/pyomo_training_convergence_vdp_new.pkl' # 32 neurons

pyomo = Results.load_results(fn)
pyomo_df = Results.collect_data_into_df(pyomo)

map_vdp = (pyomo_df.param1 == 'vdp') & (pyomo_df.param2 == True)
pyomo_df = pyomo_df[map_vdp].groupby(['param1', 'param2']).agg(list).reset_index()
pyomo_df.drop(columns=['param2'], inplace=True)
pyomo_df['time_elapsed'] = pyomo_df.time_elapsed.item()[-1]
pyomo_df_32 = pyomo_df.copy()

pyomo_df_32.rename(columns={'mse_train': 'training_loss', 'mse_test':'testing_loss'}, inplace=True)
pyomo_df_32 = GraphsTraining.prepare_timings_general(pyomo_df_32)
pyomo_df_32['type'] = 'pyomo'
pyomo_df_32

FileNotFoundError: [Errno 2] No such file or directory: 'results/pyomo_training_convergence_vdp_new.pkl'

### Load Pytorch results in

In [ ]:
fn = 'results/pytorch_training_conv_vdp_new.pkl'
fn_time = 'results/pytorch_training_conv_vdp_time_new.pkl'

pt = {True:Results.load_results(fn)}
pt_time = {True:Results.load_results(fn_time)}

pt_df = Results.collect_data_into_df(pt)
pt_time_df = Results.collect_data_into_df(pt_time)

pt_df['time_elapsed'] = pt_time_df['time_elapsed']
pt_df.rename(columns={'param1': 'pretraining', 'train_loss':'training_loss'}, inplace=True)
#display(pt_df)

pt_df = GraphsTraining.extract_training_loss(pt_df)
pt_df = GraphsTraining.split_train_test_losses(pt_df)
pt_df['pyomo_pretraining'] = False
pt_df['type'] = 'pt'
train_info.set_pretraining_time(pt_time_df['time_elapsed'].item()[0], 'pt_regular')
pt_df_regular = pt_df.copy()

pt_df_regular

### Load Pytorch results in with Pyomo pretraining

In [ ]:
fn = 'results/pytorch_training_conv_vdp_pyomo_new.pkl'
fn_time = 'results/pytorch_training_conv_vdp_time_pyomo_new.pkl'

pt = {False:Results.load_results(fn)}
pt_time = {False:Results.load_results(fn_time)}

pt_df = Results.collect_data_into_df(pt)
pt_time_df = Results.collect_data_into_df(pt_time)

pt_df['time_elapsed'] = pt_time_df['time_elapsed']
pt_df.rename(columns={'param1': 'pretraining', 'train_loss':'training_loss'}, inplace=True)
#display(pt_df)

pt_df = GraphsTraining.extract_training_loss(pt_df)
pt_df = GraphsTraining.split_train_test_losses(pt_df)
pt_df['pyomo_pretraining'] = True
pt_df['type'] = 'pt'

pt_df_pyomo = pt_df.copy()
pt_df_pyomo

### Load diffrax-jax results in

In [ ]:
fn = 'results/jax_training_conv_vdp_new.pkl'
fn_time = 'results/jax_training_conv_vdp_time_new.pkl'

jd = Results.load_results(fn)
jd_time = Results.load_results(fn_time)

jd_df = Results.collect_data_into_df(jd)
jd_time_df = Results.collect_data_into_df(jd_time)

jd_df['time_elapsed'] = jd_time_df['time_elapsed']
jd_df.rename(columns={'param2': 'pretraining'}, inplace=True)

jd_df[jd_df.pretraining]['time_elapsed'].item()[0]

In [ ]:
fn = 'results/jax_training_conv_vdp_new.pkl'
fn_time = 'results/jax_training_conv_vdp_time_new.pkl'

jd = Results.load_results(fn)
jd_time = Results.load_results(fn_time)

jd_df = Results.collect_data_into_df(jd)
jd_time_df = Results.collect_data_into_df(jd_time)

jd_df['time_elapsed'] = jd_time_df['time_elapsed']
jd_df.rename(columns={'param2': 'pretraining'}, inplace=True)

time_elpased_pretraining = jd_df[jd_df.pretraining]['time_elapsed'].item()[0]
train_info.set_pretraining_time(time_elpased_pretraining, 'regular')

jd_df = GraphsTraining.extract_training_loss(jd_df)
jd_df = GraphsTraining.split_train_test_losses(jd_df)
jd_df['type'] = 'jd'
jd_df

### Load diffrax-jax with pyomo pretraining

In [7]:
fn_pyomo = 'results/diffrax_pyomo_pretraining_32_pyomo.pkl'
fn_pyomo_time = 'results/diffrax_pyomo_pretraining_time_32_pyomo.pkl'

jd_pyomo = Results.load_results(fn_pyomo)
jd_pyomo_time = Results.load_results(fn_pyomo_time)

jd_df_pyomo = Results.collect_data_into_df(jd_pyomo)
jd_time_df_pyomo = Results.collect_data_into_df(jd_pyomo_time)
jd_df_pyomo.time_elapsed = jd_time_df_pyomo.time_elapsed
jd_df_pyomo['pretraining'] = False
jd_df_pyomo['param1'] = 'vdp'
jd_df_pyomo['type'] = 'jd'

In [ ]:
jd_df_pyomo = GraphsTraining.extract_training_loss(jd_df_pyomo)
jd_df_pyomo = GraphsTraining.split_train_test_losses(jd_df_pyomo)
display(jd_df_pyomo)

with open("../00_trained_wb/trained_wb_32.pkl", "rb") as f:
    custom_params = pickle.load(f)

jd_df['pyomo_pretraining'] = False
jd_df_pyomo['pyomo_pretraining'] = True

### Combine

In [ ]:
# concatenate the dataframes
df = pd.concat([jd_df, jd_df_pyomo, pt_df_regular, pt_df_pyomo], ignore_index=True)
df

### Save pretraining times

In [10]:
df.loc[df.pretraining, 'time_elapsed'] = df.loc[df.pretraining].time_elapsed.apply(lambda x: x[1])
train_info.set_pretraining_time(custom_params['time_elapsed'], 'pyomo')
train_info.set_pretraining_time(custom_params['time_elapsed'], 'pt_pyomo')

In [ ]:
train_info.prepare_timings(df)
df

In [ ]:
print(pyomo_df_32.training_loss.item()[-1])
print(pyomo_df_32.testing_loss.item()[-1])
print(pyomo_df_32.time_elapsed.item())

In [ ]:
n = 4
print(df.iloc[n].training_loss[-1])
print(df.iloc[n].testing_loss[-1])
print(df.iloc[n].time_elapsed)

In [ ]:
lw = 1.75
plt.figure(figsize=(10, 6))

plt.plot(df.iloc[3].times, df.iloc[3].training_loss, label='PyTorch', color='orange')
plt.plot(df.iloc[3].times[0], df.iloc[3].training_loss[0], marker = 'X', markersize = 10, label='pre-training ends', color='orange')
plt.plot(df.iloc[1].times, df.iloc[1].training_loss, label='JAX', color='green')
plt.plot(df.iloc[1].times[0], df.iloc[1].training_loss[0], marker = 'X', markersize = 10, label='pre-training ends', color='green')
plt.plot(pyomo_df_32.times.item(), pyomo_df_32.training_loss.item(), label='Pyomo', color='blue', linewidth=lw)

# plt.title('MSE loss during training (Train set) for 3 different models for the Van der Pol Oscillator')
plt.xlabel("Training Time (s)", fontsize=20)
plt.ylabel('Training MSE (log scale)', fontsize=20)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.yscale('log')
plt.legend()
plt.grid(True)

plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), shadow=False, ncol=3, fontsize=18)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(df.iloc[3].times, df.iloc[3].testing_loss, label='PyTorch', color='orange')
plt.plot(df.iloc[3].times[0], df.iloc[3].testing_loss[0], marker = 'X', markersize = 10, label='pre-training ends', color='orange')
plt.plot(df.iloc[1].times, df.iloc[1].testing_loss, label='JAX', color='green')
plt.plot(df.iloc[1].times[0], df.iloc[1].testing_loss[0], marker = 'X', markersize = 10, label='pre-training ends', color='green')
plt.plot(pyomo_df_32.times.item(), pyomo_df_32.testing_loss.item(), label='Pyomo', color='blue', linewidth=1.75)

#plt.title('MSE loss during training (Test set) for 3 different models for the Van der Pol Oscillator')
plt.xlabel("Training Time (s)", fontsize=20)
plt.ylabel('Testing MSE (log scale)', fontsize=20)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.yscale('log')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), shadow=False, ncol=3, fontsize=18)
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(df.iloc[0].times, df.iloc[0].training_loss, label='JAX (no pre-training)', color='green', ls = '--')
plt.plot(df.iloc[1].times[0], df.iloc[1].training_loss[0], marker = 'X', markersize = 10, label='Regular JAX pre-training ends', color='green')
plt.plot(df.iloc[1].times, df.iloc[1].training_loss, label='JAX (regular pre-training)', color='green')
plt.plot(pyomo_df_32.times.item(), pyomo_df_32.training_loss.item(), label='Pyomo', color='blue', ls ='--', alpha=0.8)
plt.plot(df.iloc[2].times, df.iloc[2].training_loss, label='JAX (Pyomo pre-train)', color='purple')

#plt.title('MSE loss during training (Train set) for 2 initialization methods for the Van der Pol Oscillator')
plt.xlabel("Training Time (s)", fontsize=20)
plt.ylabel('Testing MSE (log scale)', fontsize=20)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.yscale('log')

plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), shadow=False, ncol=2, fontsize=18)
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(df.iloc[1].times[0], df.iloc[1].testing_loss[0], label='diffrax-jax regular pretraining finishes', color='green', marker = 'X', markersize = 10, )
plt.plot(df.iloc[1].times, df.iloc[1].testing_loss, label='diffrax-jax after regular pretraining', color='green')
plt.plot(df.iloc[2].times, df.iloc[2].testing_loss, label='diffrax-jax after pyomo pretraining', color='purple')
plt.plot(pyomo_df_32.times.item(), pyomo_df_32.testing_loss.item(), label='pyomo model used for pretraining', color='blue', ls ='--')

plt.title('MSE loss during training (Test set) for 2 initialization methods for the Van der Pol Oscillator')
plt.xlabel('Training time [s]')
plt.ylabel('MSE loss during training (Test set) - log scale')
plt.yscale('log')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(df.iloc[3].times[0], df.iloc[3].training_loss[0], label='regular pretraining finished', color='orange', marker = 'X', markersize = 10)
plt.plot(df.iloc[3].times, df.iloc[3].training_loss, label='pytorch after regular pretraining', color='orange')
plt.plot(df.iloc[4].times, df.iloc[4].training_loss, label='pytorch after pyomo pretraining', color='orange')
plt.plot(pyomo_df_32.times.item(), pyomo_df_32.training_loss.item(), label='pyomo model used for pretraining', color='blue', ls ='--')

plt.title('MSE loss during training (Train set) for 2 initialization methods for the Van der Pol Oscillator (PyTorch)')
plt.xlabel('Training time [s]')
plt.ylabel('MSE loss during training (Train set) - log scale')
plt.yscale('log')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(df.iloc[3].times[0], df.iloc[3].testing_loss[0], label='regular pretraining finished', color='orange', marker = 'X', markersize = 10)
plt.plot(df.iloc[3].times, df.iloc[3].testing_loss, label='pytorch after regular pretraining', color='orange')
plt.plot(df.iloc[4].times, df.iloc[4].testing_loss, label='pytorch after pyomo pretraining', color='orange')
plt.plot(pyomo_df_32.times.item(), pyomo_df_32.testing_loss.item(), label='pyomo model used for pretraining', color='blue', ls ='--')

plt.title('MSE loss during training (Test set) for 2 initialization methods for the Van der Pol Oscillator (PyTorch)')
plt.xlabel('Training time [s]')
plt.ylabel('MSE loss during training (Test set) - log scale')
plt.yscale('log')
plt.legend()
plt.grid(True)
plt.show()